In [ ]:
import os
os.environ["HF_TOKEN"] =""

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
import math

torch._dynamo.reset()

torch.backends.cudnn.benchmark = True

# =============================================================================
# CONFIG
# =============================================================================

class Config:
    vocab_size     = 50257
    embedding_dim  = 512
    num_heads      = 8
    num_layers     = 6
    max_seq_len    = 256
    dropout        = 0.1

    batch_size     = 16
    learning_rate  = 3e-4
    max_steps      = 100000
    eval_every     = 500
    save_every     = 5000

    hf_token       = os.environ.get("HF_TOKEN", "") # Secured token loading
    dataset_name   = "HuggingFaceFW/fineweb-edu"
    dataset_split  = "train"
    dataset_sample = 100000

    device = "cuda" if torch.cuda.is_available() else "cpu"


# =============================================================================
# TOKENIZER
# =============================================================================

from transformers import GPT2TokenizerFast

def get_tokenizer():
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

# =============================================================================
# DATASET
# =============================================================================

class FineWebDataset(Dataset):
    def __init__(self, config, tokenizer):
        path_loader = "/kaggle/input/datasets/newmos/tokenizer787/dataset_cache.pt"

        if os.path.exists(path_loader):
            print("Loading tokenized dataset from cache...")
            self.chunks = torch.load(path_loader)
            return

        print("Loading FineWeb-Edu dataset...")
        dataset = load_dataset(
            config.dataset_name,
            name="sample-10BT",
            split=config.dataset_split,
            token=config.hf_token,
            streaming=True,
        ).take(config.dataset_sample)

        all_tokens = []
        for item in dataset:
            tokens = tokenizer.encode(item["text"], truncation=False)
            all_tokens.extend(tokens)
            all_tokens.append(tokenizer.eos_token_id)

        self.chunks =[]
        for i in range(0, len(all_tokens) - config.max_seq_len - 1, config.max_seq_len):
            chunk = all_tokens[i : i + config.max_seq_len + 1]
            self.chunks.append(torch.tensor(chunk, dtype=torch.long))

        torch.save(self.chunks, path_loader)

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        return chunk[:-1], chunk[1:]

# =============================================================================
# ROPE & COMPLEX HELPERS
# =============================================================================

def complex_dropout(x, p=0.1, training=True):
    if not training or p == 0.0:
        return x
    mask = F.dropout(torch.ones_like(x.real), p=p, training=training)
    return x * mask

def build_rope_cache(seq_len, dim, device):
    theta = 1.0 / (10000 ** (torch.arange(0, dim, device=device).float() / dim))
    positions = torch.arange(seq_len, device=device).float()
    angles = torch.outer(positions, theta)
    return torch.polar(torch.ones_like(angles), angles)

def apply_rope(x, rope_cache):
    # Slice cache to current sequence length T
    T = x.size(1)
    return x * rope_cache[:T].unsqueeze(0).unsqueeze(2)

# =============================================================================
# ENGRAM MEMORY
# =============================================================================

class EngramMemory(nn.Module):
    def __init__(self, config, num_slots=262139, ngram_orders=(1, 2, 3), num_hash_heads=4):
        super().__init__()
        self.ngram_orders  = ngram_orders
        self.num_hash_heads = num_hash_heads
        self.num_slots     = num_slots
        self.complex_dim   = config.embedding_dim // 2

        self.table = nn.Embedding(num_slots, config.embedding_dim)
        nn.init.normal_(self.table.weight, std=0.02)

        self.register_buffer(
            'coeffs',
            torch.randint(1, 100003, (len(ngram_orders), num_hash_heads))
        )

        # Vector gate: learnable weight per dimension
        self.gate = nn.Linear(self.complex_dim, self.complex_dim, bias=True)
        nn.init.constant_(self.gate.bias, -2.0) # Start closed to prioritize attention initially

    def _hash_ngrams(self, token_ids, order, n_idx, head_idx):
        B, T   = token_ids.shape
        coeff  = self.coeffs[n_idx, head_idx]
        h      = torch.zeros(B, T, device=token_ids.device, dtype=torch.long)

        for i in range(order):
            if i == 0:
                tok = token_ids
            else:
                # Use GPT-2 EOS token (50256) instead of 0 (!) for padding out-of-bounds history
                tok = torch.full_like(token_ids, 50256)
                tok[:, i:] = token_ids[:, :-i]
            h = (h * coeff + tok) % self.num_slots
        return h

    def forward(self, token_ids, x):
        B, T = token_ids.shape
    
        all_indices  = torch.stack([
            self._hash_ngrams(token_ids, order, n_idx, h)
            for n_idx, order in enumerate(self.ngram_orders)
            for h in range(self.num_hash_heads)
        ], dim=2)  # [B, T, 12]
    
        retrieved = self.table(all_indices).mean(dim=2)  # [B, T, embedding_dim]
        mem  = torch.complex(retrieved[:, :, 0::2], retrieved[:, :, 1::2])
        gate = torch.sigmoid(self.gate(x.abs()))
        return x + gate * mem

# =============================================================================
# ATTENTION & FEEDFORWARD
# =============================================================================

class ComplexMultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.complex_dim = config.embedding_dim // 2
        self.head_dim = self.complex_dim // config.num_heads

        self.W_q = nn.Linear(self.complex_dim, self.complex_dim, dtype=torch.cfloat, bias=False)
        self.W_k = nn.Linear(self.complex_dim, self.complex_dim, dtype=torch.cfloat, bias=False)
        self.W_v = nn.Linear(self.complex_dim, self.complex_dim, dtype=torch.cfloat, bias=False)
        self.W_o = nn.Linear(self.complex_dim, self.complex_dim, dtype=torch.cfloat, bias=False)
        
        # Scale complex weights to maintain variance across 4-term products
        for layer in[self.W_q, self.W_k, self.W_v, self.W_o]:
            # Initialize real and imag separately to prevent PyTorch uniform_ crash on cfloat
            nn.init.xavier_uniform_(layer.weight.data.real)
            nn.init.xavier_uniform_(layer.weight.data.imag)
            layer.weight.data.mul_(1 / math.sqrt(2))

        self.dropout = nn.Dropout(config.dropout)
        self.scale = math.sqrt(self.head_dim)

    def forward(self, x, rope_cache, mask):
        B, T, _ = x.shape

        Q = apply_rope(self.W_q(x).reshape(B, T, self.num_heads, self.head_dim), rope_cache)
        K = apply_rope(self.W_k(x).reshape(B, T, self.num_heads, self.head_dim), rope_cache)
        V = self.W_v(x).reshape(B, T, self.num_heads, self.head_dim)

        Q, K, V = Q.transpose(1, 2), K.transpose(1, 2), V.transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1).conj()) / self.scale
        scores = scores.real # Preserves symmetric cosine relative position signal
        scores = scores.masked_fill(mask[:T, :T] == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1).to(torch.cfloat)
        attn = complex_dropout(attn, p=self.dropout.p, training=self.training)

        out = torch.matmul(attn, V).transpose(1, 2).reshape(B, T, self.complex_dim)
        return self.W_o(out)

class ComplexFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        complex_dim  = config.embedding_dim // 2
        hidden_dim   = complex_dim * 4

        self.fc1 = nn.Linear(complex_dim, hidden_dim, dtype=torch.cfloat, bias=False)
        self.fc2 = nn.Linear(hidden_dim, complex_dim, dtype=torch.cfloat, bias=False)
        
        for layer in[self.fc1, self.fc2]:
            # Initialize real and imag separately to prevent PyTorch uniform_ crash on cfloat
            nn.init.xavier_uniform_(layer.weight.data.real)
            nn.init.xavier_uniform_(layer.weight.data.imag)
            layer.weight.data.mul_(1 / math.sqrt(2))
            
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc1(x)
        # modGELU: Phase-invariant activation
        mag = x.abs()
        x = F.gelu(mag) * (x / (mag + 1e-8))
        x = complex_dropout(x, p=self.dropout.p, training=self.training)
        return self.fc2(x)

# =============================================================================
# BLOCKS & FULL MODEL
# =============================================================================

class ComplexLayerNorm(nn.Module):
    def __init__(self, complex_dim):
        super().__init__()
        self.norm = nn.LayerNorm(complex_dim * 2)

    def forward(self, x):
        B, T, C = x.shape
        x_real = torch.view_as_real(x).view(B, T, C * 2)
        x_norm = self.norm(x_real).view(B, T, C, 2)
        return torch.complex(x_norm[..., 0], x_norm[..., 1])

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        complex_dim = config.embedding_dim // 2
        self.norm1     = ComplexLayerNorm(complex_dim)
        self.norm2     = ComplexLayerNorm(complex_dim)
        self.attention = ComplexMultiHeadAttention(config)
        self.ff        = ComplexFeedForward(config)
        self.dropout   = nn.Dropout(config.dropout)

    def forward(self, x, rope_cache, mask):
        x = x + complex_dropout(self.attention(self.norm1(x), rope_cache, mask), p=self.dropout.p, training=self.training)
        x = x + complex_dropout(self.ff(self.norm2(x)), p=self.dropout.p, training=self.training)
        return x

class ComplexLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        complex_dim = config.embedding_dim // 2

        self.embedding = nn.Embedding(config.vocab_size, config.embedding_dim)
        nn.init.normal_(self.embedding.weight, std=0.02) # Standardized for stability

        self.engram = EngramMemory(config)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.num_layers)])
        self.norm_final = ComplexLayerNorm(complex_dim)
        self.head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)
        self.dropout = nn.Dropout(config.dropout)

        # Pre-compute fixed-size RoPE cache
        head_dim = complex_dim // config.num_heads
        self.register_buffer("rope_cache", build_rope_cache(config.max_seq_len, head_dim, config.device))
        self.register_buffer("mask", torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)))

    def forward(self, tokens):
        B, T = tokens.shape
        x = self.embedding(tokens)
        x = torch.complex(x[:, :, 0::2], x[:, :, 1::2])
        x = complex_dropout(x, p=self.dropout.p, training=self.training)

        x = self.engram(tokens, x)

        mask = self.mask[:T, :T]
        for block in self.blocks:
            x = block(x, self.rope_cache, mask)

        x = self.norm_final(x)
        logits = self.head(torch.view_as_real(x).view(B, T, -1))
        return logits

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# =============================================================================
# LOSS & TRAINING
# =============================================================================

def unlikelihood_loss(logits, targets, input_ids, alpha=0.1, recent_window=8):
    # 1. Compute Log Softmax ONCE for the entire vocab
    log_probs = F.log_softmax(logits, dim=-1)  # [B, T, V]
    
    # 2. Base Loss (NLLLoss + log_softmax is mathematically identical to CrossEntropy)
    ce_loss = F.nll_loss(log_probs.view(-1, log_probs.size(-1)), targets.view(-1))
    
    # 3. UL Loss Setup
    pad_id = 50256
    pad    = recent_window - 1
    recent = F.pad(input_ids, (pad, 0), value=pad_id).unfold(1, recent_window, 1)

    # 4. Gather the log probabilities FIRST, then exponentiate.
    # This completely skips running a second softmax over the 50k vocabulary!
    recent_log_probs = log_probs.gather(2, recent)
    recent_probs     = torch.exp(recent_log_probs)

    # 5. Mask out the actual target token and the padding token
    is_target  = (recent == targets.unsqueeze(-1))
    is_padding = (recent == pad_id)
    valid_mask = (~is_target & ~is_padding).float()

    # 6. Calculate penalty
    ul_loss_elements = -torch.log(1.0 - recent_probs + 1e-8) * valid_mask
    num_penalized    = valid_mask.sum()
    
    if num_penalized > 0:
        ul_loss = ul_loss_elements.sum() / num_penalized
    else:
        ul_loss = torch.tensor(0.0, device=logits.device)

    return ce_loss + alpha * ul_loss


# =============================================================================
# TRAINING LOOP
# =============================================================================

def train():
    config = Config()
    model = ComplexLLM(config).to(config.device)
    model = torch.compile(model)
    tokenizer = get_tokenizer()
    dataset = FineWebDataset(config, tokenizer)
    dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, foreach=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.max_steps, eta_min=1e-5)

    step = 0

    # =========================================================================
    # RESUME FROM CHECKPOINT
    # =========================================================================
    if os.path.isdir("/kaggle/input/models/newmos/no/pytorch/default/3/checkpoints"):
        checkpoints = sorted([f for f in os.listdir("/kaggle/input/models/newmos/no/pytorch/default/3/checkpoints") if f.startswith("step_") and f.endswith(".pt")],
            key=lambda f: int(f.split("_")[1].split(".")[0])
        )
        if checkpoints:
            latest      = checkpoints[-1]
            latest_step = int(latest.split("_")[1].split(".")[0])
            print(f"Resuming from checkpoint: {latest} (step {latest_step})")
            
            # Load the weights (strict=False allows us to add/remove components without breaking)
            model.load_state_dict(torch.load(
                f"/kaggle/input/models/newmos/no/pytorch/fastcomplex-v2/1/{latest}",
                map_location=config.device,
                weights_only=True  # Security best-practice for loading .pt files
            ), strict=False)
            
            step = latest_step
            
            # Catch up the scheduler so learning rate matches the resumed step
            for _ in range(step):
                scheduler.step()
        else:
            print("No valid checkpoints found. Starting fresh.")
    else:
        print("Starting training fresh.")

    model.train()
    total_loss = 0

    for epoch in range(999):
        for x, y in dataloader:
            if step >= config.max_steps: return
            x, y = x.to(config.device), y.to(config.device)
            
            logits = model(x)
            loss = unlikelihood_loss(logits, y, x)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            step += 1

            if step % config.eval_every == 0:
                print(f"Step {step:>6} | Loss: {total_loss/config.eval_every:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
                total_loss = 0

            if step % config.save_every == 0:
                os.makedirs("checkpoints", exist_ok=True)
                torch.save(model.state_dict(), f"checkpoints/step_{step}.pt")

if __name__ == "__main__":
    train()


In [ ]:
import os

# List all files in the output directory
for dirname, _, filenames in os.walk('/kaggle/working/checkpoints/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import shutil

# Move the 10k checkpoint from the subfolder to the main output folder
shutil.copy('/kaggle/working/checkpoints/step_24000.pt', '/kaggle/working/step_24000.pt')

print("File moved to main directory. Refresh the Data pane on the right.")

In [ ]:
import os
import glob

# Get all .pt files in the checkpoints folder
all_checkpoints = glob.glob('/kaggle/working/*.pt')

for file_path in all_checkpoints:
    # Delete everything UNLESS it is the 10000 step
    if 'step_24000.pt' not in file_path:
        os.remove(file_path)
        print(f"Deleted: {file_path}")

print("\nCleanup finished. Only step_24000.pt should remain.")